In [3]:
import pandas as pd
df1 = pd.read_csv("real_flight_data_batch_1.csv")
df2 = pd.read_csv("real_flight_data_batch_2.csv")
df3 = pd.read_csv("real_flight_data_batch_3.csv")
df4 = pd.read_csv("real_flight_data_batch_4.csv")


df = pd.concat([df1, df2, df3, df4], ignore_index=True)
df.to_csv("all_flight_list.csv", index=False, encoding="utf-8-sig")

In [4]:
df.describe()


,price,number_of_stops
count,100771.000000,100771.0
mean,590.207312,0.0
std,206.478024,0.0
min,11.000000,0.0
25%,415.000000,0.0
50%,588.000000,0.0
75%,747.000000,0.0
max,999.000000,0.0


In [5]:
df.columns

Index(['origin_city', 'origin_airport_code', 'destination_city',
       'destination_airport_code', 'country', 'flight_date', 'airline',
       'price', 'currency', 'number_of_stops', 'departure_time',
       'arrival_time', 'flight_duration', 'scraped_at', 'raw_price_string'],
      dtype='object')

In [6]:
df = df.drop(columns=["number_of_stops"])

In [7]:
df.sort_values("price").head(20)[
    ["origin_city","destination_city","flight_date","airline",
     "raw_price_string","price","currency","departure_time","arrival_time","scraped_at"]
]

,origin_city,destination_city,flight_date,airline,raw_price_string,price,currency,departure_time,arrival_time,scraped_at
100770,Baku,Surabaya,2026-12-09,China Southern,$11,11.0,USD,"22:4522:45, 9 Ara Çar – 22:00+122:00, 10 Ara P...",Check Duration,2026-02-14T01:18:35.743498
100769,Baku,Surabaya,2026-12-09,China Southern,$11,11.0,USD,22:45,Check Duration,2026-02-14T01:18:35.576623
100758,Baku,Jakarta,2026-12-16,China Southern,$11,11.0,USD,"22:4522:45, 16 Ara Çar – 23:20+123:20, 17 Ara ...",Check Duration,2026-02-14T00:51:51.345084
100757,Baku,Jakarta,2026-12-16,China Southern,$11,11.0,USD,"22:4522:45, 16 Ara Çar – 23:25+123:25, 17 Ara ...",Check Duration,2026-02-14T00:51:51.267280
100756,Baku,Jakarta,2026-12-16,China Southern,$11,11.0,USD,22:45,Check Duration,2026-02-14T00:51:50.962567
100751,Baku,Jakarta,2026-12-09,China Southern,$11,11.0,USD,22:45,Check Duration,2026-02-14T00:49:43.433192
100750,Baku,Jakarta,2026-12-02,China Southern,$11,11.0,USD,"22:4522:45, 2 Ara Çar – 23:20+123:20, 3 Ara Pe...",Check Duration,2026-02-14T00:47:09.626099
100749,Baku,Jakarta,2026-12-02,China Southern,$11,11.0,USD,"22:4522:45, 2 Ara Çar – 23:25+123:25, 3 Ara Pe...",Check Duration,2026-02-14T00:47:09.591956
100748,Baku,Jakarta,2026-12-02,China Southern,$11,11.0,USD,22:45,Check Duration,2026-02-14T00:47:09.329300
12845,Baku,Tokyo,2026-03-26,"Qatar Airways, Malaysia Airlines, JAL",$11,11.0,USD,"22:0522:05, 26 Mar Per – 16:15+216:15, 28 Mar ...",Check Duration,2026-02-08T12:39:58.206790


In [8]:
# 1) Drop etibarsız sütunlar
df = df.drop(columns=["flight_duration", "number_of_stops", "arrival_time", "scraped_at"], errors="ignore")

In [9]:
# 2) Tip düzəlişi
df["flight_date"] = pd.to_datetime(df["flight_date"], errors="coerce")
df["price"] = pd.to_numeric(df["price"], errors="coerce")

In [10]:
# 3) airline_clean yarat
df["airline_clean"] = df["airline"].astype(str)
df["airline_clean"] = df["airline_clean"].str.replace(r"İşleten:.*$", "", regex=True).str.strip()
df["airline_clean"] = df["airline_clean"].str.replace(r"([a-zəöüşçı])([A-ZƏÖÜŞÇİ])", r"\1, \2", regex=True)

In [11]:
# Regex-in parçaladığı brendləri düzəlt
df["airline_clean"] = df["airline_clean"].str.replace(r"\bSun,\s*Express\b", "SunExpress", regex=True)
df["airline_clean"] = df["airline_clean"].str.replace(r"\bFly,\s*Arystan\b", "FlyArystan", regex=True)
df["airline_clean"] = df["airline_clean"].str.replace(r"\bEgypt,\s*Air\b", "EgyptAir", regex=True)
df["airline_clean"] = df["airline_clean"].str.replace(r"\bIndi,\s*Go\b", "IndiGo", regex=True)
df["airline_clean"] = df["airline_clean"].replace({"LOTAustrian": "LOT, Austrian"})
df["airline_clean"] = df["airline_clean"].str.replace(r"\s*,\s*", ", ", regex=True).str.strip()

In [13]:
# 4) departure_time fix (HH:MM çıxart)
import pandas as pd, re
time_pat = re.compile(r"\b(\d{1,2}:\d{2})\b")
def extract_time(x):
    if pd.isna(x): 
        return None
    m = time_pat.search(str(x))
    return m.group(1).zfill(5) if m else None
df["departure_time"] = df["departure_time"].apply(extract_time)

In [14]:
# 5) Kritik boşları at
df = df.dropna(subset=["flight_date", "price", "departure_time", "airline_clean"])

In [15]:
before = len(df)

df = df[df["price"] >= 50]   # 50$-dan aşağı olanları silirik

after = len(df)
print("Silinən sətir sayı (ucuz səhv qiymətlər):", before - after)
print("Qalan sətir sayı:", after)

Silinən sətir sayı (ucuz səhv qiymətlər): 514
Qalan sətir sayı: 77546


In [16]:
# 7) Dedupe (offer-level)
df = df.drop_duplicates(subset=[
    "origin_airport_code","destination_airport_code","flight_date","departure_time","airline_clean","price"
])

In [17]:
df

,origin_city,origin_airport_code,destination_city,destination_airport_code,country,flight_date,airline,price,currency,departure_time,raw_price_string,airline_clean
0,Baku,GYD,London,LHR,UK,2026-01-03,Turkish Airlines,512.0,USD,07:55,$512,Turkish Airlines
1,Baku,GYD,London,LHR,UK,2026-01-03,Turkish Airlines,512.0,USD,17:45,$512,Turkish Airlines
2,Baku,GYD,London,LHR,UK,2026-01-03,Turkish Airlines,529.0,USD,04:20,$529,Turkish Airlines
3,Baku,GYD,London,LHR,UK,2026-01-03,"Azerbaycan Hava Yolları, Turkish Airlines",756.0,USD,09:40,$756,"Azerbaycan Hava Yolları, Turkish Airlines"
4,Baku,GYD,London,LHR,UK,2026-01-03,Turkish Airlines,512.0,USD,22:05,$512,Turkish Airlines
...,...,...,...,...,...,...,...,...,...,...,...,...
100764,Baku,GYD,Surabaya,SUB,Indonesia,2026-10-23,China Southern,994.0,USD,22:00,$994,China Southern
100765,Baku,GYD,Surabaya,SUB,Indonesia,2026-11-13,China Southern,994.0,USD,22:45,$994,China Southern
100766,Baku,GYD,Surabaya,SUB,Indonesia,2026-11-13,China Southern,994.0,USD,22:00,$994,China Southern
100767,Baku,GYD,Surabaya,SUB,Indonesia,2026-11-20,China Southern,994.0,USD,22:45,$994,China Southern


In [18]:
df.columns

Index(['origin_city', 'origin_airport_code', 'destination_city',
       'destination_airport_code', 'country', 'flight_date', 'airline',
       'price', 'currency', 'departure_time', 'raw_price_string',
       'airline_clean'],
      dtype='object')

In [25]:
# 8) Final 10 sütun
final_cols = [
    "origin_city","origin_airport_code","destination_city","destination_airport_code",
    "country","flight_date","departure_time","price","currency","airline_clean"
]
df = df[final_cols]

In [26]:
# 9) Health check
print(df.shape)
print(df.isna().sum())

df.to_csv("all_flight_list.csv", index=False)

(73259, 10)
origin_city                 0
origin_airport_code         0
destination_city            0
destination_airport_code    0
country                     0
flight_date                 0
departure_time              0
price                       0
currency                    0
airline_clean               0
dtype: int64


In [24]:
df["price"].dtype

dtype('float64')

In [27]:
df

,origin_city,origin_airport_code,destination_city,destination_airport_code,country,flight_date,departure_time,price,currency,airline_clean
0,Baku,GYD,London,LHR,UK,2026-01-03,07:55,512.0,USD,Turkish Airlines
1,Baku,GYD,London,LHR,UK,2026-01-03,17:45,512.0,USD,Turkish Airlines
2,Baku,GYD,London,LHR,UK,2026-01-03,04:20,529.0,USD,Turkish Airlines
3,Baku,GYD,London,LHR,UK,2026-01-03,09:40,756.0,USD,"Azerbaycan Hava Yolları, Turkish Airlines"
4,Baku,GYD,London,LHR,UK,2026-01-03,22:05,512.0,USD,Turkish Airlines
...,...,...,...,...,...,...,...,...,...,...
100764,Baku,GYD,Surabaya,SUB,Indonesia,2026-10-23,22:00,994.0,USD,China Southern
100765,Baku,GYD,Surabaya,SUB,Indonesia,2026-11-13,22:45,994.0,USD,China Southern
100766,Baku,GYD,Surabaya,SUB,Indonesia,2026-11-13,22:00,994.0,USD,China Southern
100767,Baku,GYD,Surabaya,SUB,Indonesia,2026-11-20,22:45,994.0,USD,China Southern
